# AutoSolve ML Training Pipeline

This notebook trains and exports all machine learning models for the **AutoSolve** Blender camera tracking addon.
It is designed to run in **Google Colab** (free GPU) or locally.

### Models trained
| Model | Purpose | Output |
|---|---|---|
| **Track Quality Predictor** | Predicts whether a track will survive the next 20 frames | `track_predictor.onnx` |
| **Settings Reward Model** | Predicts expected tracking reward given footage class + parameters | `settings_model.onnx` |
| **Region Trackability Heatmap** | Empirical survival weights per screen region | `region_weights.json` |

### Runtime format
Models are exported as **ONNX** (primary) for fast inference via the bundled `onnxruntime` library,
plus a **JSON/NumPy** fallback for compatibility.
Both formats are produced automatically by the export step.

### Live data
Every AutoSolve solve in Blender automatically writes a training record to `ml/data/live/`.
These are merged into the training set in Step 2, so the model continuously improves from real usage.

---
## Step 1: Setup — Upload Project & Install Dependencies

Upload `AutoSolve.zip` to Colab, or clone from GitHub.

In [ ]:
# ── Option A: clone from GitHub (recommended) ──────────────────────────────
# !git clone https://github.com/usamasq/AutoSolve.git
# %cd AutoSolve

# ── Option B: upload a zip ─────────────────────────────────────────────────
# !unzip -q AutoSolve.zip -d AutoSolve
# %cd AutoSolve

# Verify project layout
!ls -la ml/

In [ ]:
# Install / verify dependencies
!pip install -q torch numpy onnx onnxruntime

import torch, numpy as np, onnx, onnxruntime as ort
print(f"PyTorch:      {torch.__version__}")
print(f"NumPy:        {np.__version__}")
print(f"ONNX:         {onnx.__version__}")
print(f"onnxruntime:  {ort.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")

---
## Step 2: Prepare Datasets

This step:
1. Merges any **live training records** written by AutoSolve during real Blender sessions (`ml/data/live/*.jsonl`)
2. Processes raw solve JSON files from `ml/data/raw/`
3. Produces a unified `settings_dataset.json` ready for training

If no real data exists yet, a synthetic dataset is generated automatically for bootstrapping.

In [ ]:
import os

# Create required directories
for d in ['ml/data/raw', 'ml/data/processed', 'ml/data/live', 'ml/runs']:
    os.makedirs(d, exist_ok=True)

# Show live data collected from Blender sessions
live_files = [f for f in os.listdir('ml/data/live') if f.endswith('.jsonl')]
if live_files:
    print(f"Found {len(live_files)} live data file(s):")
    for f in live_files:
        lines = sum(1 for _ in open(f'ml/data/live/{f}'))
        print(f"  {f}  ({lines} records)")
else:
    print("No live data yet — synthetic dataset will be generated.")

# Merge live data into raw/ before preparation
import shutil
for f in live_files:
    src = f'ml/data/live/{f}'
    dst = f'ml/data/raw/{f}'
    shutil.copy2(src, dst)
    print(f"Merged: {f}")

---
## Step 2a: Direct Video Feature Ingestion

Run direct video analysis on raw files placed in `ml/clips/` using OpenCV to extract motion speed, zoom divergence, noise ratio, and distortion curvature in Python.

In [ ]:
# Extract features directly from videos inside ml/clips/
!python ml/extract_video_features.py --clips-dir ml/clips --out-dir ml/data/raw

In [ ]:
# Prepare dataset (generates synthetic data if raw/ is empty)
!python ml/prepare_dataset.py \
    --data-dir ml/data/raw \
    --output-json ml/data/processed/settings_dataset.json

---
## Step 3: Train Track Quality Predictor

Trains a 3-layer MLP that predicts whether an active track will survive the next 20 frames.
Input features: patch size, motion, error history, region, clip resolution class.

In [ ]:
!python ml/train_track_predictor.py \
    --data-dir ml/data/raw \
    --out-dir ml/runs/track_predictor \
    --epochs 100

---
## Step 4: Train Settings Reward Model

Trains the expected-reward MLP that predicts how good a set of tracking parameters will perform
for a given footage class. AutoSolve uses this to rank settings candidates before tracking starts.

In [ ]:
!python ml/train_settings_model.py \
    --data-path ml/data/processed/settings_dataset.json \
    --out-dir ml/runs/settings_optimizer \
    --epochs 50

In [ ]:
# Evaluate on validation split
!python ml/evaluate_model.py \
    --data-path ml/data/processed/settings_dataset.json \
    --model-path ml/runs/settings_optimizer/model_meta_weights.json

---
## Step 5: Aggregate Region Trackability Heatmap

Builds an empirical frequency table of which screen regions produce surviving tracks,
split by footage type. Used to bias auto-placement toward historically productive zones.

In [ ]:
!python ml/train_trackability_model.py \
    --data-dir ml/data/raw \
    --output-json ml/runs/region_weights.json

---
## Step 6: Export to ONNX (Primary) + JSON Fallback

Converts trained PyTorch models into two formats:
- **ONNX** — used by the bundled `onnxruntime` inside Blender (fast, compiled)
- **JSON/NumPy** — fallback for any environment without onnxruntime

Both the `.onnx` and `_meta.json` normalisation files are required for the ONNX path.

In [ ]:
# Export both models to ONNX + validate outputs match PyTorch to <1e-3
!python ml/export_onnx.py \
    --track-weights ml/runs/track_predictor/model_meta_weights.json \
    --settings-weights ml/runs/settings_optimizer/model_meta_weights.json \
    --out-dir ml/runs/onnx

In [ ]:
# Also export JSON/NumPy fallback weights
!python ml/export_numpy_model.py \
    --input-json ml/runs/track_predictor/model_meta_weights.json \
    --output-path ml/runs/track_predictor.json

!python ml/export_defaults.py \
    --model-path ml/runs/settings_optimizer/model_meta_weights.json \
    --output-json ml/runs/recommended_defaults.json

In [ ]:
# Verify all output files exist
import os

expected = [
    'ml/runs/onnx/track_predictor.onnx',
    'ml/runs/onnx/track_predictor_meta.json',
    'ml/runs/onnx/settings_model.onnx',
    'ml/runs/onnx/settings_model_meta.json',
    'ml/runs/track_predictor.json',
    'ml/runs/region_weights.json',
    'ml/runs/recommended_defaults.json',
]

all_ok = True
for f in expected:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) // 1024 if exists else 0
    status = f'✅  {size}KB' if exists else '❌  MISSING'
    print(f'{status}  {f}')
    if not exists:
        all_ok = False

print()
print('All outputs ready ✅' if all_ok else '⚠️  Some files missing — check training steps above')

---
## Step 7: Download & Deploy

Download the exported files then copy them into the addon.

### Where each file goes

| File | Destination in addon |
|---|---|
| `track_predictor.onnx` | `autosolve/tracker/models/track_predictor.onnx` |
| `track_predictor_meta.json` | `autosolve/tracker/models/track_predictor_meta.json` |
| `settings_model.onnx` | `autosolve/tracker/models/settings_model.onnx` |
| `settings_model_meta.json` | `autosolve/tracker/models/settings_model_meta.json` |
| `track_predictor.json` | `autosolve/tracker/models/track_predictor.json` *(fallback)* |
| `region_weights.json` | `autosolve/tracker/presets/region_weights.json` |
| `recommended_defaults.json` | Merge into `PRETRAINED_DEFAULTS` in `autosolve/tracker/constants.py` |

In [ ]:
# Download all output files (Colab only)
from google.colab import files
import os

to_download = [
    'ml/runs/onnx/track_predictor.onnx',
    'ml/runs/onnx/track_predictor_meta.json',
    'ml/runs/onnx/settings_model.onnx',
    'ml/runs/onnx/settings_model_meta.json',
    'ml/runs/track_predictor.json',
    'ml/runs/region_weights.json',
    'ml/runs/recommended_defaults.json',
]

for f in to_download:
    if os.path.exists(f):
        print(f'Downloading {f}...')
        files.download(f)
    else:
        print(f'SKIP (not found): {f}')

---
## Optional: Bulk Training on Multiple Clips

If you have many raw Blender solve JSON files, you can run collection and training in batch.

In [ ]:
# List all raw data files
import os, json

raw_files = [f for f in os.listdir('ml/data/raw') if f.endswith(('.json', '.jsonl'))]
print(f'Raw data files: {len(raw_files)}')
total_records = 0
for f in raw_files:
    path = f'ml/data/raw/{f}'
    try:
        with open(path) as fh:
            if f.endswith('.jsonl'):
                n = sum(1 for _ in fh)
            else:
                data = json.load(fh)
                n = len(data) if isinstance(data, list) else 1
        total_records += n
        print(f'  {f}  ({n} records)')
    except Exception as e:
        print(f'  {f}  ERROR: {e}')

print(f'\nTotal records: {total_records}')

In [ ]:
# Re-run full pipeline in one shot (after uploading more data above)
!python ml/prepare_dataset.py --data-dir ml/data/raw --output-json ml/data/processed/settings_dataset.json
!python ml/train_track_predictor.py  --data-dir ml/data/raw --out-dir ml/runs/track_predictor --epochs 150
!python ml/train_settings_model.py   --data-path ml/data/processed/settings_dataset.json --out-dir ml/runs/settings_optimizer --epochs 75
!python ml/train_trackability_model.py --data-dir ml/data/raw --output-json ml/runs/region_weights.json
!python ml/export_onnx.py --track-weights ml/runs/track_predictor/model_meta_weights.json --settings-weights ml/runs/settings_optimizer/model_meta_weights.json --out-dir ml/runs/onnx
print('\n✅ Full pipeline complete')